## A first compose file — web + db

Here's a real two-service stack: a Postgres database and an Nginx web server.

```yaml
# compose.yaml
services:
  db:
    image: postgres:16
    environment:
      POSTGRES_PASSWORD: secret
      POSTGRES_DB: app
    volumes:
      - pgdata:/var/lib/postgresql/data

  web:
    image: nginx:alpine
    ports:
      - "8080:80"
    depends_on:
      - db

volumes:
  pgdata: {}
```

Three things to notice, each a payoff of earlier modules:

1. **No explicit network, yet they can talk.** Both services land on the auto-created `<project>_default` network, so `web` reaches `db` by the hostname **`db`** — the embedded-DNS magic from module 05, for free.
2. **A named volume.** `pgdata` is declared under top-level `volumes:` and referenced in `db` — Compose creates it on first `up`, and it persists the database (module 04) across restarts.
3. **`depends_on: db`.** Compose starts `db` before `web`. (Necessary but not *sufficient* — the next section shows why.)

Bring it up and manage it with the project-aware commands:

```bash
docker compose up -d          # create + start, detached
docker compose ps             # what's running
docker compose logs -f web    # tail web's logs
docker compose down           # stop + remove (named volumes survive)
docker compose down -v        # also wipe named volumes
```

Notice `down` keeps your volumes by default — your database survives an `up`/`down` cycle, and only `down -v` deletes it. That one file, checked into git, *is* the environment: a teammate clones and `docker compose up` gets the identical web + db stack.